##### Aprendizaje supervisado: cada observación tiene una respuesta asociada que guía el aprendizaje

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import dash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder 
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
from sklearn import tree


df  = pd.read_csv("archivos_csv/titanic.csv")

age_null_values = df["Age"].isnull()

df.info()

age_null_values.replace({True:1,False:0})

null_values = age_null_values.sum()

print("-----------------------------------------")
print(f"Valores nulos en la columna 'Age': {null_values}")

### Variables categóricas
nominales: son aquellas que actúan como etiquetas de los objetos, dentro de estas pueden denominarse binarias a las que se dividen en 2 clases. Por último, se codifican para el arpendizaje.

ordinales: son aquellas que representan un órden en particular que rige en los datos.

In [ ]:
# imputación de datos faltantes

df.fillna({"Age":df["Age"].mean()},inplace=True)

mode = df["Embarked"].mode()
df.fillna({"Embarked":mode[0]},inplace=True)

# codificando variables categóricas nominales

df["Sex_encoded"] = LabelEncoder().fit_transform(df["Sex"])

# a su vez generamos una versión textual de la variable binaria objetivo

map_survived = {
    0:"no",
    1:"yes"
}
df["Survived_no_encoded"] = df["Survived"].apply(lambda x : map_survived.get(x))

df[["Survived","Survived_no_encoded"]]

### CART(classification and regression trees)
algoritmos basados en utilizar umbrales que condicionen el destino de una variable, pero no cualquier umbral, sino aquellos que dependiendo la problemática en cestión(en este caso de clasificación) obtengan el menor valor en cuanto a la probabilidad de clasificar mal la variable objetivo o grado de aleatoriedad del umbral, se realizan repetidas pruebas de este tipo y se determian los umbrales óptimos del modelo. Otra de las características es que se modifica el volumen, es decir, la cantidad de particiones que tendrá el modelo, siendo denominado criterio de parada como el término utilizado. Para ello, hay parámetros que reciben valores y representan cierta particularidad de tamaño como la distancia que habrá entre el nodo(condición) inicial y la última hoja(valor asociado) o la suma mínima de pesos de instancia necesaria en un elemento secundario. Dependiendo de los valores que se les otorguen, el árbol predictor sufrirá de un crecimiento o reducción en cuanto a su complejidad y evitando el sobreajuste en este último caso.

In [ ]:
# seccionamos los datos en grupos de entrenamiento y prueba

x_train, x_test, y_train, y_test = train_test_split(
                                                   df[["Sex_encoded","Pclass","Fare","Parch","SibSp"]],
                                                   df["Survived"],
                                                   test_size=0.1)

# declaración del algoritmo

tree_decision = tree.DecisionTreeClassifier(criterion="entropy", # medida de aleatoriedad, determinará los nodos del modelo en base a la probabilidad de clasificar erroneamente una variable, pero sin darle mayor importancias a ciertas características
                                           max_depth=6, # distancia entre el nodo(condición) principal y la última hoja(respuesta asociada), valores recomendados: 3-10
                                           min_samples_split=6, # mínimo número de muestras requeridas para dividir un nodo, valores recomendados: 2-10
                                           min_samples_leaf=5, #  mínimo número de muestras requeridas en cada hoja, valores recomendados: 1-10
                                           ccp_alpha=0.003) # valor de post-poda(luego de que el modelo se ajute) que evita el sobreajuste 

model = tree_decision.fit(x_train, y_train)

# predicción de los datos de prueba

class_predicts = model.predict(x_test)

class_real = y_test.values

# métricas de rendimientos para clasificación

matrix_confusion = confusion_matrix(class_real,class_predicts)
TP = matrix_confusion[0,0]
FP = matrix_confusion[0,1]
FN = matrix_confusion[1,0]
TN = matrix_confusion[1,1]

accuracy = accuracy_score(class_real, class_predicts)
color_accuracy = "green"
if accuracy < 0.6:
    color_accuracy = "red"
accuracy_str = str(accuracy)

recall = recall_score(class_real, class_predicts)
color_recall = "green"
if recall < 0.6:
    color_recall = "red"
recall_str = str(recall)

precision = precision_score(class_real, class_predicts)
color_precision = "green"
if precision < 0.6:
    color_precision = "red"
precision_str = str(precision)

F1_score = f1_score(class_real, class_predicts)
color_f1 = "green"
if F1_score < 0.6:
    color_f1 = "red"
F1_score_str = str(F1_score)

print(tree.export_text(model, feature_names=["Sex_encoded","Pclass","Fare","Parch","SibSp"]))

plt.figure(figsize=(35,22))
tree.plot_tree(model, feature_names=["Sex_encoded","Pclass","Fare","Parch","SibSp"])
plt.show()

##### Dashboard que muestra los porcentages y promedios de cada variable categórica objetivo y el desempeño del modelo creado

In [ ]:
# creación de dashboard

app = dash.Dash(__name__)

app.layout = html.Div(id="body",className="e1_body",children=[
html.H1("Titanic",id="title",className="e1_title"),
html.Div(className="e1_dashboards",children=[
    html.Div(id="graph_div_1",className="e1_graph_div",children=[
        html.Div(id="dropdown_div_1",className="e1_dropdown_div",children=[
            dcc.Dropdown(id="dropdown_1",className="e1_dropdown",
                        options = [
                            {"label":"Sexo","value":"Sex"},
                            {"label":"Clase social","value":"Pclass"},
                            {"label":"Embarcadero","value":"Embarked"},
                            {"label":"Padres e hijos/as","value":"Parch"},
                            {"label":"Hermanas/os y esposas/os","value":"SibSp"},
                        ],
                        value="Sex",
                        multi=False,
                        clearable=False)
        ]),
        dcc.Graph(id="piechart",className="e1_graph",figure={})
    ]),
    html.Div(id="graph_div_2",className="e1_graph_div",children=[
        html.Div(id="dropdown_div_2",className="e1_dropdown_div",children=[
            dcc.Dropdown(id="dropdown_2",className="e1_dropdown",
                        options = [
                            {"label":"Edad","value":"Age"},
                            {"label":"Boleto","value":"Fare"},
                        ],
                        value="Age",
                        multi=False,
                        clearable=False)
        ]),
        dcc.Graph(id="bar",className="e1_graph",figure={})
    ]),
]),
    
    html.Div(className="e1_div",children=[
        html.Div(id="rendimiento",className="e1_rendimiento",children=[
            html.P([html.B("Clases reales",style={"color":"blue"}),"   |   ",html.B("Predicciones",style={"color":"red"})],style={"text-align":"center","font-family":"sans-serif"}),
            html.P("----------------------------------------------------------------------------------------------------",style={"margin":"0"}),
            html.P(f"{class_real}",className="e1_clases_reales"),
            html.P(f"{class_predicts}",className="e1_predicciones")
        ]),
        html.Div(id="modelos_metricas",className="e1_metricas",children=[
                html.P("Matriz de confusión",style={"font-size":"0.9em","text-align":"center","font-family":"sans-serif","font-weigth":"bold"}),
                html.Div(className="e1_matriz",id="matriz_confusion",children=[
                html.Div([html.B(TP,style={"color":"green","font-family":"sans-serif"})],id="TP",className="e1_aciertos"), 
                html.Div([html.B(FP,style={"color":"red","font-family":"sans-serif"})],id="FP",className="e1_aciertos"),
                html.Div([html.B(FN,style={"color":"red","font-family":"sans-serif"})],id="FN",className="e1_aciertos"),
                html.Div([html.B(TN,style={"color":"green","font-family":"sans-serif"})],id="TN",className="e1_aciertos")
                ]),
                html.Div(className="e1_puntuaciones",children=[
                html.Ul(id="lista",children=[
                html.Li([f"Accuracy: ",html.B(accuracy_str[:4],style={"color":f"{color_accuracy}"})],id="accuracy",style={"font-family":"sans-serif","margin-right":"5px"}),
                html.Li([f"Recall: ",html.B(recall_str[:4],style={"color":f"{color_recall}"})],id="recall",style={"font-family":"sans-serif","margin-right":"5px"}),
                html.Li([f"Precision: ",html.B(precision_str[:4],style={"color":f"{color_precision}"})],id="precision",style={"font-family":"sans-serif","margin-right":"5px"}),
                html.Li([f"F1 Score: ",html.B(F1_score_str[:4],style={"color":f"{color_f1}"})],id="f1_score",style={"font-family":"sans-serif","margin-right":"5px"})
                ])
                
            ])
        ])
    ])
])

# generan interacción entre los datos de entrada y los elementos que serán modificados 

@app.callback(
    [Output(component_id="piechart",component_property="figure"),
    Output(component_id="bar",component_property="figure")],
    [Input(component_id="dropdown_1",component_property="value"),
    Input(component_id="dropdown_2",component_property="value")]
)

# función que se ejecuta cada vez que se interactúa con los elementos 

def update_graph(slct_var_cat,slct_var_num):
    
    df_percentage = df.groupby(slct_var_cat)["Survived"].mean()
    df_percentage = df_percentage.reset_index()
    df_percentage["Survived"] = df_percentage["Survived"] * 100
    
    piechart = px.pie(df_percentage,values='Survived',names=slct_var_cat,title='Porcentage de sobrevivir')
    
    
    df_mean = df.groupby("Survived_no_encoded")[slct_var_num].mean()
    df_mean = df_mean.reset_index()
    
    barplot = px.bar(df_mean,x="Survived_no_encoded",y=slct_var_num,title='Medias de Edad y Boleto',labels={"x":"sobrevivientes","y":slct_var_num})
    barplot.update_layout(xaxis_title="Sobrevivientes")
    
    return piechart,barplot

if __name__ == "__main__":
    app.run_server(debug=False)

#### Error de varianza

es la variabilidad que existe en la función objetivo con respecto a los diferentes datos de entrenamiento que se utilicen para la creación del modelo, sucede principalmente en algoritmos que se ajustan fácilmente a los datos y requieren menos suposiciones a diferencia de los algoritmos con alto bías, algunos ejemplos de algoritmos con alta variranza son: KNN, CART o Naive Bayes

In [ ]:
accuracies = []
bucle = list(range(14))

for i in bucle:
    x_train, x_test, y_train, y_test = train_test_split(
                                                   df[["Sex_encoded","Pclass","Fare","Parch","SibSp"]],
                                                   df["Survived"],
                                                   test_size=0.25)
    class_real = y_test.values
    class_predicts = model.predict(x_test)
    accuracy = accuracy_score(class_real, class_predicts)
    accuracies.append(accuracy)
    
plt.plot(bucle, accuracies, marker="*", c="r")
plt.grid("on")
plt.title("Error de Varianza")
plt.ylabel("Acuracies")
plt.show()